# Local Feature Importance Demo

This notebook demonstrates the proximity-based local feature importance added to `forestgap.core`. The example uses binary features so the perturbation step matches the bit-flip setup described in *Explicating Feature Contribution using Random Forest Proximity Distances*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier

from forestgap import with_sparse_proximity

In [ ]:
rng = np.random.RandomState(321)
X = rng.randint(0, 2, size=(160, 3)).astype(float)
y = X[:, 0].astype(int)

columns = ["signal", "noise_1", "noise_2"]
X_train = pd.DataFrame(X[:120], columns=columns)
X_test = pd.DataFrame(X[120:], columns=columns)
y_train = y[:120]
y_test = y[120:]

In [ ]:
ProxRandomForestClassifier = with_sparse_proximity(RandomForestClassifier)

model = ProxRandomForestClassifier(
    n_estimators=80,
    random_state=21,
    prox_method="original",
    matrix_type="sparse",
    bootstrap=False,
    max_features=None,
)
model.fit(X_train, y_train)

importance, closeness = model.local_feature_importance(
    X_test.iloc[:5],
    labels=y_test[:5],
    perturbation="flip",
    return_closeness=True,
)

summary = importance.copy()
summary.insert(0, "true_label", y_test[:5])
summary.insert(1, "predicted_label", model.predict(X_test.iloc[:5]))
summary

In [ ]:
sample_id = importance.index[0]
sample_scores = importance.loc[sample_id].sort_values(key=np.abs, ascending=False)

ax = sample_scores.plot(kind="bar", color=["#1f77b4", "#9ecae1", "#9ecae1"])
ax.axhline(0.0, color="black", linewidth=1)
ax.set_ylabel("Local contribution")
ax.set_title(f"Sample {sample_id}: local feature importance")
plt.tight_layout()
plt.show()

closeness["in_group"].loc[[sample_id]]